# Movements in Movements Chart

Real Interest Rate vs. Investment (GFCF) for Brazil and peer countries, 2000–2024.

Also known as the "Fan Belt" effect visualization.

In [106]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np
from plotly.colors import hex_to_rgb

# ── Helpers ──
def lighten(hex_color, factor):
    """Blend hex_color toward white by factor (0=original, 1=white)."""
    r, g, b = hex_to_rgb(hex_color)
    return f'rgb({int(r + (255 - r) * factor)},{int(g + (255 - g) * factor)},{int(b + (255 - b) * factor)})'

def darken(hex_color, factor):
    """Blend hex_color toward black by factor (0=original, 1=black)."""
    r, g, b = hex_to_rgb(hex_color)
    return f'rgb({int(r * (1 - factor))},{int(g * (1 - factor))},{int(b * (1 - factor))})'

def color_gradient(hex_color, n_segments):
    """Return n_segments colors from light→dark for a base hex color."""
    if n_segments <= 1:
        return [hex_color]
    colors = []
    for i in range(n_segments):
        t = i / (n_segments - 1)  # 0→1
        # Go from 55% lightened to 15% darkened
        if t < 0.5:
            colors.append(lighten(hex_color, 0.55 - t * 1.1))
        else:
            colors.append(darken(hex_color, (t - 0.5) * 0.3))
    return colors

# ── Data ──
df_all = pd.read_stata("../Data/Course_Database_Newest.dta")
df_all = df_all.rename(columns={'countrycodeiso': 'countrycode'})

PEERS = ['COL', 'ARG', 'MEX', 'ZAF', 'CHL', 'URY']
PEER_NAMES = {
    'BRA': 'Brazil', 'COL': 'Colombia', 'ARG': 'Argentina',
    'MEX': 'Mexico', 'ZAF': 'South Africa', 'CHL': 'Chile', 'URY': 'Uruguay'
}
PEER_COLORS = {
    'BRA': '#e36209',
    'COL': '#a67b5b', 'ARG': '#a67b5b',
    'MEX': '#7b2d8e', 'ZAF': '#7b2d8e',
    'CHL': '#2c7bb6', 'URY': '#2c7bb6',
}
PEER_SYMBOLS = {
    'BRA': 'circle', 'COL': 'square', 'ARG': 'cross',
    'MEX': 'diamond', 'ZAF': 'x', 'CHL': 'triangle-up', 'URY': 'triangle-down',
}

gfcf_var = 'wdi_ne_gdi_ftot_zs'
rir_var  = 'wdi_fr_inr_rinr'
codes = ['BRA'] + PEERS

sdata = df_all[df_all['countrycode'].isin(codes)].copy()
sdata = sdata[sdata['year'] >= 2000].dropna(subset=[gfcf_var, rir_var])
sdata = sdata.sort_values(['countrycode', 'year'])

# ── Per-country data ──
country_data = {}
for code in codes:
    cdf = sdata[sdata['countrycode'] == code].sort_values('year')
    if cdf.empty:
        continue
    country_data[code] = {
        'xs': cdf[gfcf_var].values,
        'ys': cdf[rir_var].values,
        'yrs': cdf['year'].astype(int).values,
    }

all_xs = sdata[gfcf_var].values
all_ys = sdata[rir_var].values
pad_x = (all_xs.max() - all_xs.min()) * 0.08
pad_y = (all_ys.max() - all_ys.min()) * 0.08
global_xrange = [all_xs.min() - pad_x, all_xs.max() + pad_x]
global_yrange = [all_ys.min() - pad_y, all_ys.max() + pad_y]

valid_codes = [c for c in codes if c in country_data]
N = len(valid_codes)

In [107]:
# ══════════════════════════════════════════════════════════════
# TRACE STRUCTURE per country:
#   [0] scatter markers (always visible)
#   [1] year-label text layer (hidden; toggled by button)
#   [2..2+n_seg-1] gradient line segments (hidden; toggled)
#
# So per country: 2 + n_segments traces
# ══════════════════════════════════════════════════════════════

fig = go.Figure()

# Track trace indices per country for button visibility toggling
trace_map = {}  # code -> {'scatter': idx, 'text': idx, 'segments': [idx, ...]}
trace_idx = 0

for code in valid_codes:
    cd    = country_data[code]
    name  = PEER_NAMES[code]
    color = PEER_COLORS[code]
    sym   = PEER_SYMBOLS[code]
    xs, ys, yrs = cd['xs'], cd['ys'], cd['yrs']
    n_pts = len(xs)
    n_seg = max(n_pts - 1, 1)
    grad  = color_gradient(color, n_seg)

    indices = {'scatter': trace_idx, 'text': None, 'segments': []}

    # ── Scatter markers ──
    fig.add_trace(go.Scatter(
        x=xs, y=ys, mode='markers',
        marker=dict(
            color=color,
            size=14 if code == 'BRA' else 11,
            symbol=sym,
            opacity=0.92 if code == 'BRA' else 0.78,
            line=dict(color='white', width=1),
        ),
        name=code,
        showlegend=True,
        hovertemplate=(
            "<b style='font-size:15px'>%{customdata[0]}</b><br>"
            "<span style='font-size:13px'>"
            "Year: <b>%{customdata[1]}</b><br>"
            "GFCF: <b>%{x:.1f}%</b> of GDP<br>"
            "Real Interest Rate: <b>%{y:.1f}%</b>"
            "</span><extra></extra>"
        ),
        customdata=list(zip([name]*n_pts, yrs.astype(str))),
    ))
    trace_idx += 1

    # ── Year-label text overlay ──
    indices['text'] = trace_idx
    fig.add_trace(go.Scatter(
        x=xs, y=ys, mode='markers+text',
        marker=dict(color=color, size=14, symbol=sym,
                    line=dict(color='white', width=1)),
        text=[f"<b>{y}</b>" for y in yrs],
        textposition='top center',
        textfont=dict(size=15, color=color, family='Arial Black'),
        name=code + ' years',
        showlegend=False,
        hoverinfo='skip',
        visible=False,
    ))
    trace_idx += 1

    # ── Gradient line segments (one trace per segment) ──
    for s in range(n_seg):
        seg_color = grad[s]
        indices['segments'].append(trace_idx)
        fig.add_trace(go.Scatter(
            x=[xs[s], xs[s+1]] if s < n_pts - 1 else [xs[s]],
            y=[ys[s], ys[s+1]] if s < n_pts - 1 else [ys[s]],
            mode='lines',
            line=dict(color=seg_color, width=2.5, shape='linear'),
            showlegend=False,
            hoverinfo='skip',
            visible=False,
        ))
        trace_idx += 1

    trace_map[code] = indices

total_traces = trace_idx

In [108]:
# ══════════════════════════════════════════════════════════════
# BUTTONS
# ══════════════════════════════════════════════════════════════
buttons = []

# "Show All": only scatter traces visible
vis_all = [False] * total_traces
for code in valid_codes:
    vis_all[trace_map[code]['scatter']] = True
buttons.append(dict(
    label='  All  ',
    method='update',
    args=[
        {'visible': vis_all},
        {
            'xaxis.range': global_xrange,
            'yaxis.range': global_yrange,
            'annotations': [],
        }
    ],
))

# Per-country buttons
for code in valid_codes:
    cd = country_data[code]
    xs, ys = cd['xs'], cd['ys']
    name  = PEER_NAMES[code]

    vis = [False] * total_traces
    # Show all scatter traces
    for c2 in valid_codes:
        vis[trace_map[c2]['scatter']] = True
    # Show this country's text + gradient segments
    vis[trace_map[code]['text']] = True
    for seg_idx in trace_map[code]['segments']:
        vis[seg_idx] = True

    pad_cx = (xs.max() - xs.min()) * 0.18 + 0.5
    pad_cy = (ys.max() - ys.min()) * 0.18 + 1.0
    xr = [xs.min() - pad_cx, xs.max() + pad_cx]
    yr = [ys.min() - pad_cy, ys.max() + pad_cy]

    buttons.append(dict(
        label=f' {code} ',
        method='update',
        args=[
            {'visible': vis},
            {
                'xaxis.range': xr,
                'yaxis.range': yr,
                'annotations': [],
            },
        ],
    ))

In [109]:
# ══════════════════════════════════════════════════════════════
# LAYOUT (responsive for iframe embedding)
# ══════════════════════════════════════════════════════════════
fig.update_layout(
    template='plotly_white',
    autosize=True,
    margin=dict(l=60, r=30, t=100, b=95),
    plot_bgcolor='rgba(252,252,255,1)',
    paper_bgcolor='white',
    font=dict(family='Roboto Condensed, sans-serif', color='#1a1a2e'),
    xaxis=dict(
        title=dict(
            text='Gross Fixed Capital Formation (% of GDP)',
            font=dict(size=19, family='Roboto Condensed, sans-serif', color='#222'),
            standoff=12,
        ),
        tickfont=dict(size=16, color='#333'),
        ticksuffix='%',
        range=global_xrange,
        showgrid=True,
        gridcolor='rgba(180,180,200,0.35)',
        gridwidth=1,
        griddash='dot',
        zeroline=True,
        zerolinecolor='rgba(100,100,120,0.4)',
        zerolinewidth=1.5,
        showline=True,
        linecolor='#444',
        linewidth=1.5,
        mirror=False,
        ticks='outside',
        ticklen=5,
        tickcolor='#888',
        minor=dict(showgrid=True, gridcolor='rgba(200,200,210,0.2)', griddash='dot'),
    ),
    yaxis=dict(
        title=dict(
            text='Real Interest Rate (%)',
            font=dict(size=19, family='Roboto Condensed, sans-serif', color='#222'),
            standoff=12,
        ),
        tickfont=dict(size=16, color='#333'),
        ticksuffix='%',
        range=global_yrange,
        showgrid=True,
        gridcolor='rgba(180,180,200,0.35)',
        gridwidth=1,
        griddash='dot',
        zeroline=True,
        zerolinecolor='rgba(220,60,60,0.35)',
        zerolinewidth=2,
        showline=True,
        linecolor='#444',
        linewidth=1.5,
        mirror=False,
        ticks='outside',
        ticklen=5,
        tickcolor='#888',
        minor=dict(showgrid=True, gridcolor='rgba(200,200,210,0.2)', griddash='dot'),
    ),
    title=dict(
        text=(
            '<b>The "Fan Belt" Effect</b>'
            '<br><span style="font-size:17px;color:#555">'
            'Real Interest Rate vs. Investment, 2000–2024</span>'
        ),
        font=dict(size=23, family='Roboto Condensed, sans-serif', color='#1a1a2e'),
        x=0.5, xanchor='center',
        y=0.96,
    ),
    legend=dict(
        orientation='h',
        yanchor='top', y=-0.12,
        xanchor='center', x=0.5,
        font=dict(size=14, family='Roboto Condensed, sans-serif'),
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(200,200,210,0.4)',
        borderwidth=1,
        itemsizing='constant',
        tracegroupgap=4,
    ),
    hovermode='closest',
    hoverlabel=dict(
        font_size=16,
        font_family='Roboto Condensed, sans-serif',
        font_color='black',
        bgcolor='rgba(255,255,255,0.95)',
        bordercolor='#bbb',
    ),
    updatemenus=[dict(
        type='buttons',
        direction='right',
        x=0.5, xanchor='center',
        y=1.0, yanchor='bottom',
        bgcolor='rgba(245,245,250,0.95)',
        bordercolor='rgba(180,180,200,0.6)',
        borderwidth=1,
        font=dict(size=15, family='Roboto Condensed, sans-serif', color='#1a1a2e'),
        buttons=buttons,
        pad=dict(r=6, t=6, b=6, l=6),
    )],
    annotations=[],
)

fig.show()

# Save as HTML - strip inline dimensions and use JS to resize
import re

plot_div = fig.to_html(include_plotlyjs=True, full_html=False, config={'responsive': True})

# Remove the inline width/height style from the plotly div
plot_div = re.sub(r'style="height:\s*\d+px;\s*width:\s*\d+px;"', 'style="width:100%; height:100%;"', plot_div)

html_content = f'''<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        html, body {{
            width: 100%;
            height: 100%;
            overflow: hidden;
        }}
        .js-plotly-plot {{
            width: 100% !important;
            height: 100% !important;
        }}
    </style>
</head>
<body>
{plot_div}
<script>
    function resizePlot() {{
        var gd = document.querySelector('.js-plotly-plot');
        if (gd && window.Plotly) {{
            var w = window.innerWidth;
            var h = window.innerHeight;
            Plotly.relayout(gd, {{
                width: w,
                height: h
            }});
        }}
    }}
    window.addEventListener('load', function() {{
        resizePlot();
        setTimeout(resizePlot, 50);
        setTimeout(resizePlot, 200);
    }});
    window.addEventListener('resize', resizePlot);
</script>
</body>
</html>
'''

with open("mov-in-mov.html", "w", encoding="utf-8") as f:
    f.write(html_content)
    
print("HTML saved. Chart will resize to fill iframe.")

HTML saved. Chart will resize to fill iframe.


# Manufacturing Value Added as Share of GDP

Premature deindustrialisation visualization for Brazil and peer countries, 2000–present.

In [111]:
import pandas as pd
import plotly.graph_objects as go
import re

# ── Data ──
df_manuf = pd.read_stata("../Data/Course_Database_Newest.dta")
df_manuf = df_manuf.rename(columns={'countrycodeiso': 'countrycode'})

manuf_var = 'wdi_nv_ind_manf_zs'
PEERS_M = ['COL', 'ARG', 'MEX', 'ZAF', 'CHL', 'URY']
all_codes_m = ['BRA'] + PEERS_M

PEER_NAMES_M = {
    'BRA': 'Brazil', 'COL': 'Colombia', 'ARG': 'Argentina',
    'MEX': 'Mexico', 'ZAF': 'South Africa', 'CHL': 'Chile', 'URY': 'Uruguay'
}
PEER_COLORS_M = {
    'BRA': '#e36209',
    'COL': '#a67b5b', 'ARG': '#a67b5b',
    'MEX': '#7b2d8e', 'ZAF': '#7b2d8e',
    'CHL': '#2c7bb6', 'URY': '#2c7bb6',
}
PEER_SYMBOLS_M = {
    'BRA': 'circle', 'COL': 'square', 'ARG': 'cross',
    'MEX': 'diamond', 'ZAF': 'x', 'CHL': 'triangle-up', 'URY': 'triangle-down',
}
sdata_m = df_manuf[df_manuf['countrycode'].isin(all_codes_m) & (df_manuf['year'] >= 2000)].copy()
sdata_m = sdata_m.dropna(subset=[manuf_var]).sort_values(['countrycode', 'year'])

fig_manuf = go.Figure()

for code in all_codes_m:
    cdf = sdata_m[sdata_m['countrycode'] == code].sort_values('year')
    if cdf.empty:
        continue
    name = PEER_NAMES_M[code]
    color = PEER_COLORS_M[code]
    sym = PEER_SYMBOLS_M[code]
    lw = 3.5 if code == 'BRA' else 1.8
    sz = 14 if code == 'BRA' else 10

    fig_manuf.add_trace(go.Scatter(
        x=cdf['year'], y=cdf[manuf_var],
        mode='lines+markers',
        line=dict(color=color, width=lw),
        marker=dict(
            color=color, size=sz, symbol=sym,
            line=dict(color='white', width=0.8),
        ),
        name=code,
        hovertemplate=(
            f"<b style='font-size:15px'>{name}</b><br>"
            "<span style='font-size:13px'>"
            "Year: <b>%{x}</b><br>"
            "Manufacturing VA: <b>%{y:.1f}%</b> of GDP"
            "</span><extra></extra>"
        ),
    ))

# ── Layout (responsive for iframe embedding) ──
fig_manuf.update_layout(
    template='plotly_white',
    autosize=True,
    margin=dict(l=60, r=30, t=100, b=100),
    plot_bgcolor='rgba(252,252,255,1)',
    paper_bgcolor='white',
    font=dict(family='Roboto Condensed, sans-serif', color='#1a1a2e'),
    title=dict(
        text=(
            '<b>Manufacturing Value Added as Share of GDP</b>'
            '<br><span style="font-size:17px;color:#555">'
            'Premature Deindustrialisation, 2000–present</span>'
        ),
        font=dict(size=23, family='Roboto Condensed, sans-serif', color='#1a1a2e'),
        x=0.5, xanchor='center',
        y=0.96,
    ),
    xaxis=dict(
        title=dict(text='Year', font=dict(size=19, family='Roboto Condensed, sans-serif', color='#222')),
        tickfont=dict(size=16, color='#333'),
        showgrid=True, gridcolor='rgba(180,180,200,0.35)', griddash='dot',
        showline=True, linecolor='#444', linewidth=1.5,
        ticks='outside', ticklen=5, tickcolor='#888',
    ),
    yaxis=dict(
        title=dict(text='Manufacturing VA (% of GDP)', font=dict(size=19, family='Roboto Condensed, sans-serif', color='#222')),
        tickfont=dict(size=16, color='#333'),
        ticksuffix='%',
        showgrid=True, gridcolor='rgba(180,180,200,0.35)', griddash='dot',
        showline=True, linecolor='#444', linewidth=1.5,
        ticks='outside', ticklen=5, tickcolor='#888',
    ),
    legend=dict(
        orientation='h',
        yanchor='top', y=-0.17,
        xanchor='center', x=0.5,
        font=dict(size=14, family='Roboto Condensed, sans-serif'),
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(200,200,210,0.4)',
        borderwidth=1,
        itemsizing='constant',
    ),
    hovermode='closest',
    hoverlabel=dict(
        font_size=16,
        font_family='Roboto Condensed, sans-serif',
        font_color='black',
        bgcolor='rgba(255,255,255,0.95)',
        bordercolor='#bbb',
    ),
    annotations=[],
)

fig_manuf.show()

# Save as HTML with click-to-highlight functionality
plot_div_m = fig_manuf.to_html(include_plotlyjs=True, full_html=False, config={'responsive': True})
plot_div_m = re.sub(r'style="height:\s*\d+px;\s*width:\s*\d+px;"', 'style="width:100%; height:100%;"', plot_div_m)

html_content_m = f'''<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        html, body {{
            width: 100%;
            height: 100%;
            overflow: hidden;
        }}
        .js-plotly-plot {{
            width: 100% !important;
            height: 100% !important;
        }}
    </style>
</head>
<body>
{plot_div_m}
<script>
    var selectedTrace = null;
    
    function resizePlot() {{
        var gd = document.querySelector('.js-plotly-plot');
        if (gd && window.Plotly) {{
            Plotly.relayout(gd, {{
                width: window.innerWidth,
                height: window.innerHeight
            }});
        }}
    }}
    
    function setupClickHighlight() {{
        var gd = document.querySelector('.js-plotly-plot');
        if (!gd) return;
        
        var numTraces = gd.data.length;
        
        // Click on a point to highlight that trace
        gd.on('plotly_click', function(data) {{
            var clickedTrace = data.points[0].curveNumber;
            
            // If clicking same trace, reset all
            if (selectedTrace === clickedTrace) {{
                selectedTrace = null;
                var opacities = [];
                for (var i = 0; i < numTraces; i++) {{
                    opacities.push(1);
                }}
                Plotly.restyle(gd, 'opacity', [opacities]);
            }} else {{
                // Highlight clicked, dim others
                selectedTrace = clickedTrace;
                var opacities = [];
                for (var i = 0; i < numTraces; i++) {{
                    opacities.push(i === clickedTrace ? 1 : 0.15);
                }}
                Plotly.restyle(gd, 'opacity', [opacities]);
            }}
        }});
        
        // Double-click to reset
        gd.on('plotly_doubleclick', function() {{
            selectedTrace = null;
            var opacities = [];
            for (var i = 0; i < numTraces; i++) {{
                opacities.push(1);
            }}
            Plotly.restyle(gd, 'opacity', [opacities]);
        }});
    }}
    
    window.addEventListener('load', function() {{
        resizePlot();
        setTimeout(function() {{
            resizePlot();
            setupClickHighlight();
        }}, 200);
    }});
    window.addEventListener('resize', resizePlot);
</script>
</body>
</html>
'''

with open("manuf-va.html", "w", encoding="utf-8") as f:
    f.write(html_content_m)
    
print("Manufacturing VA chart saved to manuf-va.html")

Manufacturing VA chart saved to manuf-va.html


# Real GDP per Capita

Brazil and peer countries, 1960–present.

In [112]:
import pandas as pd
import plotly.graph_objects as go
import re

# ── Data ──
df_gdp = pd.read_stata("../Data/Course_Database_Newest.dta")
df_gdp = df_gdp.rename(columns={'countrycodeiso': 'countrycode'})

gdp_var = 'wdi_ny_gdp_pcap_kd'
PEERS_G = ['COL', 'ARG', 'MEX', 'ZAF', 'CHL', 'URY']
all_codes_g = ['BRA'] + PEERS_G

PEER_NAMES_G = {
    'BRA': 'Brazil', 'COL': 'Colombia', 'ARG': 'Argentina',
    'MEX': 'Mexico', 'ZAF': 'South Africa', 'CHL': 'Chile', 'URY': 'Uruguay'
}
PEER_COLORS_G = {
    'BRA': '#e36209',
    'COL': '#a67b5b', 'ARG': '#a67b5b',
    'MEX': '#7b2d8e', 'ZAF': '#7b2d8e',
    'CHL': '#2c7bb6', 'URY': '#2c7bb6',
}
PEER_SYMBOLS_G = {
    'BRA': 'circle', 'COL': 'square', 'ARG': 'cross',
    'MEX': 'diamond', 'ZAF': 'x', 'CHL': 'triangle-up', 'URY': 'triangle-down',
}

sdata_g = df_gdp[df_gdp['countrycode'].isin(all_codes_g) & (df_gdp['year'] >= 1960)].copy()
sdata_g = sdata_g.dropna(subset=[gdp_var]).sort_values(['countrycode', 'year'])

fig_gdp = go.Figure()

for code in all_codes_g:
    cdf = sdata_g[sdata_g['countrycode'] == code].sort_values('year')
    if cdf.empty:
        continue
    name = PEER_NAMES_G[code]
    color = PEER_COLORS_G[code]
    sym = PEER_SYMBOLS_G[code]
    lw = 3.5 if code == 'BRA' else 1.8
    sz = 14 if code == 'BRA' else 10

    fig_gdp.add_trace(go.Scatter(
        x=cdf['year'], y=cdf[gdp_var],
        mode='lines+markers',
        line=dict(color=color, width=lw),
        marker=dict(
            color=color, size=sz, symbol=sym,
            line=dict(color='white', width=0.8),
        ),
        name=code,
        hovertemplate=(
            f"<b style='font-size:15px'>{name}</b><br>"
            "<span style='font-size:13px'>"
            "Year: <b>%{x}</b><br>"
            "GDP/cap: <b>$%{y:,.0f}</b>"
            "</span><extra></extra>"
        ),
    ))

# ── Layout (responsive for iframe embedding) ──
fig_gdp.update_layout(
    template='plotly_white',
    autosize=True,
    margin=dict(l=60, r=30, t=100, b=100),
    plot_bgcolor='rgba(252,252,255,1)',
    paper_bgcolor='white',
    font=dict(family='Roboto Condensed, sans-serif', color='#1a1a2e'),
    title=dict(
        text=(
            '<b>Real GDP per Capita</b>'
            '<br><span style="font-size:17px;color:#555">'
            'Constant 2015 USD, 1960–present</span>'
        ),
        font=dict(size=23, family='Roboto Condensed, sans-serif', color='#1a1a2e'),
        x=0.5, xanchor='center',
        y=0.96,
    ),
    xaxis=dict(
        title=dict(text='Year', font=dict(size=19, family='Roboto Condensed, sans-serif', color='#222')),
        tickfont=dict(size=16, color='#333'),
        showgrid=True, gridcolor='rgba(180,180,200,0.35)', griddash='dot',
        showline=True, linecolor='#444', linewidth=1.5,
        ticks='outside', ticklen=5, tickcolor='#888',
        dtick=10,
    ),
    yaxis=dict(
        title=dict(text='GDP/cap (constant 2015 USD)', font=dict(size=19, family='Roboto Condensed, sans-serif', color='#222')),
        tickfont=dict(size=16, color='#333'),
        tickformat='$,.0f',
        showgrid=True, gridcolor='rgba(180,180,200,0.35)', griddash='dot',
        showline=True, linecolor='#444', linewidth=1.5,
        ticks='outside', ticklen=5, tickcolor='#888',
    ),
    legend=dict(
        orientation='h',
        yanchor='top', y=-0.12,
        xanchor='center', x=0.5,
        font=dict(size=14, family='Roboto Condensed, sans-serif'),
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(200,200,210,0.4)',
        borderwidth=1,
        itemsizing='constant',
    ),
    hovermode='closest',
    hoverlabel=dict(
        font_size=16,
        font_family='Roboto Condensed, sans-serif',
        font_color='black',
        bgcolor='rgba(255,255,255,0.95)',
        bordercolor='#bbb',
    ),
    annotations=[],
)

fig_gdp.show()

# Save as HTML with click-to-highlight functionality
plot_div_g = fig_gdp.to_html(include_plotlyjs=True, full_html=False, config={'responsive': True})
plot_div_g = re.sub(r'style="height:\s*\d+px;\s*width:\s*\d+px;"', 'style="width:100%; height:100%;"', plot_div_g)

html_content_g = f'''<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        html, body {{
            width: 100%;
            height: 100%;
            overflow: hidden;
        }}
        .js-plotly-plot {{
            width: 100% !important;
            height: 100% !important;
        }}
    </style>
</head>
<body>
{plot_div_g}
<script>
    var selectedTrace = null;
    
    function resizePlot() {{
        var gd = document.querySelector('.js-plotly-plot');
        if (gd && window.Plotly) {{
            Plotly.relayout(gd, {{
                width: window.innerWidth,
                height: window.innerHeight
            }});
        }}
    }}
    
    function setupClickHighlight() {{
        var gd = document.querySelector('.js-plotly-plot');
        if (!gd) return;
        
        var numTraces = gd.data.length;
        
        // Click on a point to highlight that trace
        gd.on('plotly_click', function(data) {{
            var clickedTrace = data.points[0].curveNumber;
            
            // If clicking same trace, reset all
            if (selectedTrace === clickedTrace) {{
                selectedTrace = null;
                var opacities = [];
                for (var i = 0; i < numTraces; i++) {{
                    opacities.push(1);
                }}
                Plotly.restyle(gd, 'opacity', [opacities]);
            }} else {{
                // Highlight clicked, dim others
                selectedTrace = clickedTrace;
                var opacities = [];
                for (var i = 0; i < numTraces; i++) {{
                    opacities.push(i === clickedTrace ? 1 : 0.15);
                }}
                Plotly.restyle(gd, 'opacity', [opacities]);
            }}
        }});
        
        // Double-click to reset
        gd.on('plotly_doubleclick', function() {{
            selectedTrace = null;
            var opacities = [];
            for (var i = 0; i < numTraces; i++) {{
                opacities.push(1);
            }}
            Plotly.restyle(gd, 'opacity', [opacities]);
        }});
    }}
    
    window.addEventListener('load', function() {{
        resizePlot();
        setTimeout(function() {{
            resizePlot();
            setupClickHighlight();
        }}, 200);
    }});
    window.addEventListener('resize', resizePlot);
</script>
</body>
</html>
'''

with open("gdp-per-capita.html", "w", encoding="utf-8") as f:
    f.write(html_content_g)
    
print("Real GDP per Capita chart saved to gdp-per-capita.html")

Real GDP per Capita chart saved to gdp-per-capita.html


# Brazil GDP Growth with Structural Breaks

Historical Overview: Brazil GDP per capita and growth with structural break detection, 1960–2024.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from scipy import stats
import re

# ── Data ──
df_brk = pd.read_stata("../Data/Course_Database_Newest.dta")
df_brk = df_brk.rename(columns={'countrycodeiso': 'countrycode'})
brazil = df_brk[df_brk['countrycode'] == 'BRA'].copy()
brazil = brazil[(brazil['year'] >= 1960) & (brazil['year'] <= 2024)].sort_values('year')
brazil = brazil.dropna(subset=['wdi_ny_gdp_pcap_kd_zg', 'wdi_ny_gdp_pcap_kd'])

years = brazil['year'].values
growth = brazil['wdi_ny_gdp_pcap_kd_zg'].values
gdp_level = brazil['wdi_ny_gdp_pcap_kd'].values

# ── Structural break detection (Chow test / sup-Wald) ──
def estat_sbsingle(y, x, trim=0.15):
    n = len(y)
    trim_obs = int(n * trim)
    best_f, best_year = 0, None
    for i in range(trim_obs, n - trim_obs):
        y1, x1 = y[:i], x[:i]
        y2, x2 = y[i:], x[i:]
        X_full = np.column_stack([np.ones(n), x])
        beta_full = np.linalg.lstsq(X_full, y, rcond=None)[0]
        ssr_full = np.sum((y - X_full @ beta_full)**2)
        X1 = np.column_stack([np.ones(len(y1)), x1])
        X2 = np.column_stack([np.ones(len(y2)), x2])
        beta1 = np.linalg.lstsq(X1, y1, rcond=None)[0]
        beta2 = np.linalg.lstsq(X2, y2, rcond=None)[0]
        ssr_split = np.sum((y1 - X1 @ beta1)**2) + np.sum((y2 - X2 @ beta2)**2)
        k = 2
        f = ((ssr_full - ssr_split) / k) / (ssr_split / (n - 2*k))
        if f > best_f:
            best_f = f
            best_year = x[i]
    return {'break_year': best_year, 'f_stat': best_f}

ANDREWS_CV = {0.01: 12.16, 0.025: 10.40, 0.05: 8.85, 0.10: 7.17}

def find_breaks_iterative(y, x, alpha):
    breaks = []
    start_idx = 0
    for _ in range(5):
        cy, cx = y[start_idx:], x[start_idx:]
        if len(cy) < 15:
            break
        result = estat_sbsingle(cy, cx)
        f_crit = stats.f.ppf(1 - alpha, 2, len(cy) - 4)
        if result['f_stat'] > f_crit:
            breaks.append(result['break_year'])
            start_idx = int(np.searchsorted(x, result['break_year'])) + 1
        else:
            break
    return sorted(breaks)

def find_breaks_andrews(y, x, alpha=0.05):
    cv = ANDREWS_CV.get(alpha, ANDREWS_CV[0.05])
    breaks = []
    start_idx = 0
    for _ in range(5):
        cy, cx = y[start_idx:], x[start_idx:]
        if len(cy) < 15:
            break
        result = estat_sbsingle(cy, cx)
        if result['f_stat'] > cv:
            breaks.append(result['break_year'])
            start_idx = int(np.searchsorted(x, result['break_year'])) + 1
        else:
            break
    return sorted(breaks)

# ── Pre-compute breaks ──
andrews_breaks = find_breaks_andrews(growth, years, alpha=0.05)
chow_levels = [0.01, 0.015, 0.02]
chow_labels = ['1%', '1.5%', '2%']
chow_breaks = {a: find_breaks_iterative(growth, years, a) for a in chow_levels}

slider_entries = []
slider_entries.append(('Andrews sup-Wald (5%)', andrews_breaks))
for a, lab in zip(chow_levels, chow_labels):
    slider_entries.append((f'Iterative Chow α={lab}', chow_breaks[a]))

# ── Build figure with 2 subplots ──
fig_brk = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.18,
    row_heights=[0.5, 0.5],
    subplot_titles=[
        '<b>Real GDP per Capita</b><br><span style="font-size:15px;color:#555">Constant 2015 USD</span>',
        '<b>GDP per Capita Growth</b><br><span style="font-size:15px;color:#555">Annual %</span>',
    ],
)

# Panel 1: GDP level line
fig_brk.add_trace(go.Scatter(
    x=years, y=gdp_level, mode='lines',
    line=dict(color='#2c7bb6', width=2.5),
    name='GDP/cap level',
    hovertemplate='<b>%{x}</b><br>GDP/cap: $%{y:,.0f}<extra></extra>',
), row=1, col=1)

# Panel 2: Growth bar chart
bar_colors = ['#d7191c' if g < 0 else '#2c7bb6' for g in growth]
fig_brk.add_trace(go.Bar(
    x=years, y=growth,
    marker_color=bar_colors, marker_line_width=0,
    name='Growth %',
    hovertemplate='<b>%{x}</b><br>Growth: %{y:.1f}%<extra></extra>',
), row=2, col=1)

fig_brk.add_hline(y=0, line_dash='solid', line_color='rgba(0,0,0,0.3)', line_width=1, row=2, col=1)

# ── Y-axis ranges ──
y1_lo = float(np.nanmin(gdp_level)) * 0.90
y1_hi = float(np.nanmax(gdp_level)) * 1.08
y2_lo = float(np.nanmin(growth)) - 2
y2_hi = float(np.nanmax(growth)) + 2

# ── Style subplot titles ──
for ann in fig_brk.layout.annotations:
    if hasattr(ann, 'text') and ann.text and '<b>' in str(ann.text):
        ann.font = dict(size=19, family='Roboto Condensed, sans-serif', color='#1a1a2e')
        if 'Real GDP' in ann.text:
            ann.y = ann.y + 0.04
        elif 'Growth' in ann.text:
            ann.y = ann.y - 0.05
base_annotations = list(fig_brk.layout.annotations)

# ── Collect all unique break years ──
all_break_years = sorted(set(by for _, brks in slider_entries for by in brks))

# ── Add break line traces ──
for by in all_break_years:
    fig_brk.add_trace(go.Scatter(
        x=[by, by], y=[y1_lo, y1_hi], mode='lines',
        line=dict(color='rgba(200,40,40,0.7)', width=2, dash='dot'),
        showlegend=False, hoverinfo='skip', visible=False,
    ), row=1, col=1)
    fig_brk.add_trace(go.Scatter(
        x=[by, by], y=[y2_lo, y2_hi], mode='lines',
        line=dict(color='rgba(200,40,40,0.7)', width=2, dash='dot'),
        showlegend=False, hoverinfo='skip', visible=False,
    ), row=2, col=1)

n_break_traces = len(fig_brk.data)

# ── Regime shading ──
regime_colors = ['#1b9e77', '#d95f02', '#7570b3', '#e7298a', '#66a61e']
x_min = float(min(years)) - 1
x_max = float(max(years)) + 1

# 1-break shading
one_break_entry = None
for label, brks in slider_entries:
    if len(brks) == 1:
        one_break_entry = (label, brks)
        break

n_1brk_shading = 0
if one_break_entry:
    edges_1 = [x_min] + sorted(one_break_entry[1]) + [x_max]
    colors_1 = ['#1b9e77', '#d95f02']
    for i in range(len(edges_1) - 1):
        x0, x1 = edges_1[i], edges_1[i+1]
        color = colors_1[i]
        fig_brk.add_trace(go.Scatter(
            x=[x0, x1, x1, x0, x0], y=[y1_lo, y1_lo, y1_hi, y1_hi, y1_lo],
            fill='toself', fillcolor=color, opacity=0.10,
            line=dict(width=0), mode='lines',
            showlegend=False, hoverinfo='skip', visible=False,
        ), row=1, col=1)
        fig_brk.add_trace(go.Scatter(
            x=[x0, x1, x1, x0, x0], y=[y2_lo, y2_lo, y2_hi, y2_hi, y2_lo],
            fill='toself', fillcolor=color, opacity=0.10,
            line=dict(width=0), mode='lines',
            showlegend=False, hoverinfo='skip', visible=False,
        ), row=2, col=1)
    n_1brk_shading = len(fig_brk.data) - n_break_traces

# 4-break shading
n_4brk_start = len(fig_brk.data)
four_break_entry = None
for label, brks in slider_entries:
    if len(brks) == 4:
        four_break_entry = (label, brks)
        break

n_4brk_shading = 0
if four_break_entry:
    edges = [x_min] + sorted(four_break_entry[1]) + [x_max]
    for i in range(len(edges) - 1):
        x0, x1 = edges[i], edges[i+1]
        color = regime_colors[i % len(regime_colors)]
        fig_brk.add_trace(go.Scatter(
            x=[x0, x1, x1, x0, x0], y=[y1_lo, y1_lo, y1_hi, y1_hi, y1_lo],
            fill='toself', fillcolor=color, opacity=0.10,
            line=dict(width=0), mode='lines',
            showlegend=False, hoverinfo='skip', visible=False,
        ), row=1, col=1)
        fig_brk.add_trace(go.Scatter(
            x=[x0, x1, x1, x0, x0], y=[y2_lo, y2_lo, y2_hi, y2_hi, y2_lo],
            fill='toself', fillcolor=color, opacity=0.10,
            line=dict(width=0), mode='lines',
            showlegend=False, hoverinfo='skip', visible=False,
        ), row=2, col=1)
    n_4brk_shading = len(fig_brk.data) - n_4brk_start

# ── Helper: regime avg growth annotations ──
def make_avg_annots(brks):
    edges = [float(min(years))] + sorted(brks) + [float(max(years)) + 1]
    annots = []
    for i in range(len(edges) - 1):
        mask = (years >= edges[i]) & (years < edges[i+1])
        if np.sum(mask) > 0:
            avg = float(np.nanmean(growth[mask]))
            mid_x = (edges[i] + edges[i+1]) / 2
            annots.append(dict(
                x=mid_x, y=y1_hi, xref='x', yref='y',
                text=f'<b>Avg={avg:.1f}%</b>',
                showarrow=False, yshift=-18,
                font=dict(size=13, color='#555', family='Roboto Condensed, sans-serif'),
            ))
    return annots

# ── Build slider steps ──
steps = []
for label, brks in slider_entries:
    vis = [True, True]
    for by in all_break_years:
        show = by in brks
        vis.append(show)
        vis.append(show)
    vis.extend([len(brks) == 1] * n_1brk_shading)
    vis.extend([len(brks) == 4] * n_4brk_shading)

    annots = list(base_annotations)
    for by in brks:
        annots.append(dict(
            x=by, y=y1_hi, xref='x', yref='y',
            text=f'<b>{int(by)}</b>', showarrow=False,
            font=dict(size=16, color='darkred', family='Roboto Condensed, sans-serif'),
            yshift=14,
        ))
    annots.extend(make_avg_annots(brks))

    steps.append(dict(
        method='update',
        args=[{'visible': vis}, {'annotations': annots}],
        label=label,
    ))

# ── Set default state ──
default_brks = slider_entries[-1][1]
for i, by in enumerate(all_break_years):
    trace_idx = 2 + i * 2
    show = by in default_brks
    fig_brk.data[trace_idx].visible = show
    fig_brk.data[trace_idx + 1].visible = show

for j in range(n_1brk_shading):
    fig_brk.data[n_break_traces + j].visible = (len(default_brks) == 1)
for j in range(n_4brk_shading):
    fig_brk.data[n_4brk_start + j].visible = (len(default_brks) == 4)

default_annots = list(base_annotations)
for by in default_brks:
    default_annots.append(dict(
        x=by, y=y1_hi, xref='x', yref='y',
        text=f'<b>{int(by)}</b>', showarrow=False,
        font=dict(size=16, color='darkred', family='Roboto Condensed, sans-serif'),
        yshift=14,
    ))
default_annots.extend(make_avg_annots(default_brks))

sliders = [dict(
    active=len(steps) - 1,
    currentvalue=dict(
        prefix='Method: ',
        font=dict(size=14, family='Roboto Condensed, sans-serif', color='#1a1a2e'),
        xanchor='center',
        offset=15,
    ),
    pad=dict(t=80),
    len=0.6,
    x=0.2,
    xanchor='left',
    steps=steps,
    font=dict(size=12, family='Roboto Condensed, sans-serif'),
)]

fig_brk.update_layout(
    template='plotly_white',
    autosize=True,
    margin=dict(l=60, r=30, t=90, b=60),
    font=dict(family='Roboto Condensed, sans-serif', color='#1a1a2e'),
    plot_bgcolor='rgba(252,252,255,1)',
    paper_bgcolor='white',
    showlegend=False,
    annotations=default_annots,
    sliders=sliders,
    xaxis=dict(range=[min(years) - 1, max(years) + 1], autorange=False),
    xaxis2=dict(
        title=dict(text='Year', font=dict(size=19, family='Roboto Condensed, sans-serif', color='#222')),
        tickfont=dict(size=16, color='#333'),
        showgrid=True, gridcolor='rgba(180,180,200,0.35)', griddash='dot',
        showline=True, linecolor='#444', linewidth=1.5,
        ticks='outside', ticklen=5, tickcolor='#888',
        dtick=10,
        range=[min(years) - 1, max(years) + 1], autorange=False,
    ),
    yaxis=dict(
        title=dict(text='GDP/cap (USD)', font=dict(size=19, family='Roboto Condensed, sans-serif', color='#222')),
        tickfont=dict(size=16, color='#333'),
        tickformat='$,.0f',
        showgrid=True, gridcolor='rgba(180,180,200,0.35)', griddash='dot',
        showline=True, linecolor='#444', linewidth=1.5,
        ticks='outside', ticklen=5, tickcolor='#888',
        range=[y1_lo, y1_hi], autorange=False,
    ),
    yaxis2=dict(
        title=dict(text='Growth (%)', font=dict(size=19, family='Roboto Condensed, sans-serif', color='#222')),
        tickfont=dict(size=16, color='#333'),
        ticksuffix='%',
        showgrid=True, gridcolor='rgba(180,180,200,0.35)', griddash='dot',
        showline=True, linecolor='#444', linewidth=1.5,
        ticks='outside', ticklen=5, tickcolor='#888',
        zeroline=True, zerolinecolor='rgba(0,0,0,0.3)', zerolinewidth=1,
        range=[y2_lo, y2_hi], autorange=False,
    ),
    hoverlabel=dict(
        font_size=16, font_family='Roboto Condensed, sans-serif', font_color='black',
        bgcolor='rgba(255,255,255,0.95)', bordercolor='#bbb',
    ),
)

fig_brk.show()

# Save as HTML
plot_div_brk = fig_brk.to_html(include_plotlyjs=True, full_html=False, config={'responsive': True})
plot_div_brk = re.sub(r'style="height:\s*\d+px;\s*width:\s*\d+px;"', 'style="width:100%; height:100%;"', plot_div_brk)

html_content_brk = f'''<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        html, body {{
            width: 100%;
            height: 100%;
            overflow: hidden;
        }}
        .js-plotly-plot {{
            width: 100% !important;
            height: 100% !important;
        }}
    </style>
</head>
<body>
{plot_div_brk}
<script>
    function resizePlot() {{
        var gd = document.querySelector('.js-plotly-plot');
        if (gd && window.Plotly) {{
            Plotly.relayout(gd, {{
                width: window.innerWidth,
                height: window.innerHeight
            }});
        }}
    }}
    window.addEventListener('load', function() {{
        resizePlot();
        setTimeout(resizePlot, 50);
        setTimeout(resizePlot, 200);
    }});
    window.addEventListener('resize', resizePlot);
</script>
</body>
</html>
'''

with open("structural-breaks.html", "w", encoding="utf-8") as f:
    f.write(html_content_brk)
    
print("Structural breaks chart saved to structural-breaks.html")

Structural breaks chart saved to structural-breaks.html
